# Работа 1. Распознавание геометрических фигур на PyTorch

Задача: обучить нейронную сеть для распознавания трёх типов геометрических фигур: треугольник, круг, квадрат.

Проверяются 18 комбинаций параметров:

- количество нейронов: 10, 100, 5000;
- функция активации: `relu`, `linear`;
- размер пакета: 10, 100, 1000.

В конце выводится итоговая таблица и один общий график для сравнения моделей.

In [ ]:
# Импорт библиотек
import os
import zipfile
import copy

import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

# Фиксируем случайность, чтобы результаты были более стабильными
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Используем видеокарту, если она доступна
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство:', device)

In [ ]:
# Загрузка базы изображений
url = 'https://storage.yandexcloud.net/aiueducation/Content/base/l3/hw_light.zip'
zip_path = 'hw_light.zip'

if not os.path.exists(zip_path):
    gdown.download(url, zip_path, quiet=False)

# Распаковка архива
if not os.path.exists('/content/hw_light') and not os.path.exists('hw_light'):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('.')

# Путь к базе
if os.path.exists('/content/hw_light'):
    base_dir = '/content/hw_light'
else:
    base_dir = 'hw_light'

print('Папка с базой:', base_dir)
print('Папки классов:', sorted(os.listdir(base_dir)))

In [ ]:
# Подготовка изображений
# В PyTorch изображения должны иметь формат: количество, каналы, высота, ширина

img_height = 20
img_width = 20

x_data = []
y_data = []

# В исходном задании классы задаются по названиям папок:
# папка 0 -> класс 0
# папка 3 -> класс 1
# остальные папки -> класс 2
for folder_name in sorted(os.listdir(base_dir)):
    folder_path = os.path.join(base_dir, folder_name)

    if not os.path.isdir(folder_path):
        continue

    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)

        # Загружаем изображение в оттенках серого и приводим к 20x20
        img = Image.open(file_path).convert('L').resize((img_width, img_height))
        img_array = np.array(img, dtype=np.float32) / 255.0

        # Добавляем размерность канала: 1x20x20
        img_array = np.expand_dims(img_array, axis=0)
        x_data.append(img_array)

        if folder_name == '0':
            y_data.append(0)
        elif folder_name == '3':
            y_data.append(1)
        else:
            y_data.append(2)

# Преобразуем данные в тензоры PyTorch
x_data = torch.tensor(np.array(x_data), dtype=torch.float32)
y_data = torch.tensor(np.array(y_data), dtype=torch.long)

print('Размер x_data:', x_data.shape)
print('Размер y_data:', y_data.shape)
print('Классы и количество:', torch.unique(y_data, return_counts=True))

# Создаём общий набор данных
dataset = TensorDataset(x_data, y_data)

# Делим данные на обучение и проверку
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_STATE)
)

print('Изображений для обучения:', len(train_dataset))
print('Изображений для проверки:', len(val_dataset))

In [ ]:
# Класс нейронной сети
class ShapeNet(nn.Module):
    def __init__(self, neurons, activation):
        super().__init__()

        # Преобразование изображения 1x20x20 в вектор из 400 чисел
        self.flatten = nn.Flatten()

        # Скрытый слой
        self.fc1 = nn.Linear(20 * 20, neurons)

        # Функция активации
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'linear':
            # linear означает отсутствие нелинейной функции активации
            self.activation = nn.Identity()
        else:
            raise ValueError('Неизвестная функция активации')

        # Выходной слой на 3 класса
        self.fc2 = nn.Linear(neurons, 3)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        return x


# Функция одной эпохи обучения или проверки
def run_epoch(model, data_loader, loss_function, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    total_correct = 0
    total_count = 0

    with torch.set_grad_enabled(is_train):
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            predictions = torch.argmax(outputs, dim=1)
            total_correct += (predictions == labels).sum().item()
            total_loss += loss.item() * images.size(0)
            total_count += images.size(0)

    average_loss = total_loss / total_count
    accuracy = total_correct / total_count

    return average_loss, accuracy

In [ ]:
# Запуск всех 18 экспериментов
EPOCHS = 10
learning_rate = 0.001

neurons_list = [10, 100, 5000]
activation_list = ['relu', 'linear']
batch_size_list = [10, 100, 1000]

results = []
histories = []

best_val_accuracy = -1
best_model_state = None
best_model_info = None

for neurons in neurons_list:
    for activation in activation_list:
        for batch_size in batch_size_list:

            print()
            print(f'Модель: нейроны = {neurons}, функция = {activation}, размер пакета = {batch_size}')

            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

            model = ShapeNet(neurons, activation).to(device)

            # В PyTorch CrossEntropyLoss уже включает softmax внутри расчёта ошибки.
            # Поэтому softmax в последнем слое модели отдельно не ставится.
            loss_function = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

            history = {
                'train_loss': [],
                'train_accuracy': [],
                'val_loss': [],
                'val_accuracy': []
            }

            for epoch in range(EPOCHS):
                train_loss, train_accuracy = run_epoch(model, train_loader, loss_function, optimizer)
                val_loss, val_accuracy = run_epoch(model, val_loader, loss_function)

                history['train_loss'].append(train_loss)
                history['train_accuracy'].append(train_accuracy)
                history['val_loss'].append(val_loss)
                history['val_accuracy'].append(val_accuracy)

                print(
                    f'Эпоха {epoch + 1}/{EPOCHS} | '
                    f'ошибка обучения: {train_loss:.4f} | '
                    f'точность обучения: {train_accuracy:.4f} | '
                    f'ошибка проверки: {val_loss:.4f} | '
                    f'точность проверки: {val_accuracy:.4f}'
                )

            # Итоговые значения берём по последней эпохе
            final_train_loss = history['train_loss'][-1]
            final_train_accuracy = history['train_accuracy'][-1]
            final_val_loss = history['val_loss'][-1]
            final_val_accuracy = history['val_accuracy'][-1]

            results.append([
                neurons,
                activation,
                batch_size,
                final_train_loss,
                final_train_accuracy,
                final_val_loss,
                final_val_accuracy
            ])

            histories.append({
                'neurons': neurons,
                'activation': activation,
                'batch_size': batch_size,
                'history': history
            })

            # Сохраняем лучшую модель по точности проверки
            if final_val_accuracy > best_val_accuracy:
                best_val_accuracy = final_val_accuracy
                best_model_state = copy.deepcopy(model.state_dict())
                best_model_info = {
                    'neurons': neurons,
                    'activation': activation,
                    'batch_size': batch_size,
                    'val_accuracy': final_val_accuracy
                }

In [ ]:
# Итоговая сравнительная таблица
results_df = pd.DataFrame(
    results,
    columns=[
        'Нейроны',
        'Активация',
        'Размер пакета',
        'Ошибка обучения',
        'Точность обучения',
        'Ошибка проверки',
        'Точность проверки'
    ]
)

# Округляем значения, чтобы таблица была читаемой
results_df_rounded = results_df.copy()
for column in ['Ошибка обучения', 'Точность обучения', 'Ошибка проверки', 'Точность проверки']:
    results_df_rounded[column] = results_df_rounded[column].round(4)

display(results_df_rounded)

print('Лучшая модель:')
print(best_model_info)

In [ ]:
# Один общий график в конце
# Здесь сравниваются все модели по проверочной выборке.

plt.figure(figsize=(18, 6))
epochs_range = range(1, EPOCHS + 1)

# Общий график точности
plt.subplot(1, 2, 1)
for item in histories:
    label_name = f'{item["neurons"]}, {item["activation"]}, пакет={item["batch_size"]}'
    plt.plot(epochs_range, item['history']['val_accuracy'], label=label_name)

plt.title('Общий график точности проверки')
plt.xlabel('Эпоха')
plt.ylabel('Точность')
plt.grid(True)
plt.legend(fontsize=7)

# Общий график ошибки
plt.subplot(1, 2, 2)
for item in histories:
    label_name = f'{item["neurons"]}, {item["activation"]}, пакет={item["batch_size"]}'
    plt.plot(epochs_range, item['history']['val_loss'], label=label_name)

plt.title('Общий график функции потерь на проверке')
plt.xlabel('Эпоха')
plt.ylabel('Ошибка')
plt.grid(True)
plt.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Сохранение лучшей модели и таблицы результатов
if best_model_state is not None:
    torch.save(
        {
            'model_state_dict': best_model_state,
            'model_info': best_model_info
        },
        'best_shape_model_pytorch.pt'
    )

results_df.to_csv('results_pytorch.csv', index=False)

print('Файл модели сохранён: best_shape_model_pytorch.pt')
print('Таблица результатов сохранена: results_pytorch.csv')

In [ ]:
# Скачивание файлов в Google Colab
# Запускайте эту ячейку после выполнения всех экспериментов.

from google.colab import files

files.download('best_shape_model_pytorch.pt')
files.download('results_pytorch.csv')